# Notebook 13 — Analyse détaillée des échecs de localisation

## Contexte

La localisation actuelle atteint Top-1 = 11.9% avec les traces seules.
Ce notebook explore les échecs pour identifier des patterns exploitables
et créer des signatures spécialisées par type de panne.

## Objectifs

1. Comprendre pourquoi ts-contacts (panne return) n'apparaît pas dans le top-5
2. Identifier les logs Java exploitables pour la panne exception
3. Analyser les patterns de latence pour network_delay
4. Proposer des améliorations concrètes

In [1]:
import pickle
import csv
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import timedelta
from collections import Counter, defaultdict
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

PROJET    = Path('/home/eunice/Bureau/Train_ticket/Intelligent_observability')
NORMAL    = PROJET / 'data/normal'
ANOMALIES = PROJET / 'data/anomalies'
MODELS_DIR = PROJET / 'models'
OUTPUT     = PROJET / 'output'
RESULTS    = PROJET / 'results'
FIGURES    = PROJET / 'figures/localisation_avancee'
FIGURES.mkdir(parents=True, exist_ok=True)

DATES_TT = ['2023-01-29', '2023-01-30']
FEATURES_SPAN = ['duration_ms']

# ─── Charger le ground truth ───
gt = pd.read_csv(OUTPUT / 'ground_truth.csv')
print(f"Ground truth : {len(gt)} fenêtres")
print(f"Types de pannes : {sorted(gt['fault_type'].unique())}")
print(f"\nRépartition :")
print(gt['fault_type'].value_counts())

# ─── Charger les modèles pré-entraînés ───
with open(MODELS_DIR / 'lof_tt.pkl', 'rb') as f:
    lof_data = pickle.load(f)
    modeles_lof = lof_data['modeles']
    scalers_lof = lof_data['scalers']

with open(MODELS_DIR / 'tfidf_tt.pkl', 'rb') as f:
    tfidf_data = pickle.load(f)
    tfidf = tfidf_data['vectorizer']
    vecteur_ref = tfidf_data['vecteur_ref']

with open(MODELS_DIR / 'if_traces_tt.pkl', 'rb') as f:
    if_data = pickle.load(f)
    modeles_if = if_data['modeles']
    scalers_if = if_data['scalers']

print(f"\n✓ Modèles chargés :")
print(f"  LOF     : {len(modeles_lof)} services")
print(f"  TF-IDF  : {len(tfidf.vocabulary_)} termes")
print(f"  IF      : {len(modeles_if)} services")

# ─── Fonctions de chargement ───
def charger_logs(date, source, fenetre):
    chemin = source / date / 'log' / f'{fenetre}_log.csv'
    if not chemin.exists():
        return pd.DataFrame()
    colonnes = ['Timestamp','TimeUnixNano','Node','PodName',
                'Container','TraceID','SpanID','Log']
    rows = []
    with open(chemin, 'r', encoding='utf-8', errors='replace') as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) >= 8:
                rows.append(row[:8])
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows, columns=colonnes)
    df['service'] = df['PodName'].apply(lambda x: str(x).rsplit('-', 2)[0])
    return df

def charger_traces(date, source, fenetre):
    chemin = source / date / 'trace' / f'{fenetre}_trace.csv'
    if not chemin.exists():
        return pd.DataFrame()
    df = pd.read_csv(chemin, on_bad_lines='skip')
    df['duration_ms'] = pd.to_numeric(df['Duration'], errors='coerce') / 1e6
    df['service'] = df['PodName'].apply(lambda x: str(x).rsplit('-', 2)[0])
    return df

print("\n✓ Fonctions prêtes")

Ground truth : 135 fenêtres
Types de pannes : ['cpu_contention', 'exception', 'network_delay', 'return']

Répartition :
fault_type
network_delay     42
exception         39
return            33
cpu_contention    21
Name: count, dtype: int64

✓ Modèles chargés :
  LOF     : 46 services
  TF-IDF  : 355 termes
  IF      : 28 services

✓ Fonctions prêtes


## 2. Analyse par type de panne — quelles fenêtres échouent ?

On mesure pour chaque type de panne :
- Où se trouve le vrai coupable dans le ranking actuel
- Quels services sont fréquemment détectés à tort en top-1
- Y a-t-il des patterns exploitables pour améliorer

In [2]:
# 
# LOCALISATION ACTUELLE (traces + fusion RRF)
# 

def localiser_traces(df_fen):
    """Ranking par IF sur spans (méthode actuelle)."""
    scores = {}
    for service in df_fen['service'].unique():
        if service not in modeles_if:
            continue
        df_svc = df_fen[df_fen['service'] == service][FEATURES_SPAN].dropna()
        if df_svc.empty:
            continue
        X = scalers_if[service].transform(df_svc)
        pred = modeles_if[service].predict(X)
        scores[service] = (pred == -1).sum() / len(pred)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# 
# ANALYSE PAR TYPE DE PANNE
# 
print("=== Analyse détaillée par type de panne ===\n")

analyses = defaultdict(list)

for _, row in gt.iterrows():
    df_traces = charger_traces(row['date'], ANOMALIES, row['window'])
    if df_traces.empty:
        continue
    
    ranking = localiser_traces(df_traces)
    services = [s for s, _ in ranking]
    vrai = row['faulty_service']
    
    if vrai in services:
        rang = services.index(vrai) + 1
    else:
        rang = -1  # non trouvé
    
    analyses[row['fault_type']].append({
        'window'    : row['window'],
        'vrai'      : vrai,
        'rang_vrai' : rang,
        'top1_predit': services[0] if services else None,
        'ranking'   : services[:5],
    })

# 
# STATISTIQUES PAR TYPE
# 
for fault_type, cas in analyses.items():
    n = len(cas)
    rangs_trouves = [c['rang_vrai'] for c in cas if c['rang_vrai'] > 0]
    non_trouves = sum(1 for c in cas if c['rang_vrai'] == -1)
    
    print(f"\n{'='*60}")
    print(f"Type : {fault_type}  ({n} fenêtres)")
    print(f"{'='*60}")
    
    if rangs_trouves:
        print(f"  Position moyenne du vrai coupable : {np.mean(rangs_trouves):.1f}")
        print(f"  Position médiane                  : {np.median(rangs_trouves):.0f}")
        print(f"  Top-1 (rang 1)                    : {sum(1 for r in rangs_trouves if r==1)}/{n}")
        print(f"  Top-3 (rang ≤ 3)                  : {sum(1 for r in rangs_trouves if r<=3)}/{n}")
        print(f"  Top-10 (rang ≤ 10)                : {sum(1 for r in rangs_trouves if r<=10)}/{n}")
    print(f"  Non trouvé dans le ranking        : {non_trouves}/{n}")
    
    # Services top-1 les plus fréquents (à tort)
    predits_top1 = [c['top1_predit'] for c in cas if c['top1_predit'] and c['top1_predit'] != c['vrai']]
    top_faux = Counter(predits_top1).most_common(5)
    if top_faux:
        print(f"\n  Services les plus souvent prédits à tort en top-1 :")
        for s, count in top_faux:
            print(f"    {s:<40} : {count} fois")

# 
# CONCLUSION
# 
print(f"\n\n{'='*60}")
print("CONCLUSION DE L'ANALYSE")
print(f"{'='*60}")

# Diagnostic
total_top10 = sum(
    sum(1 for c in cas if c['rang_vrai'] > 0 and c['rang_vrai'] <= 10)
    for cas in analyses.values()
)
print(f"\n  Vrai coupable dans top-10 : {total_top10}/135 = {total_top10/135*100:.1f}%")
print(f"  → Si on améliore le RE-RANKING du top-10, potentiel gros gain")

=== Analyse détaillée par type de panne ===


Type : return  (33 fenêtres)
  Position moyenne du vrai coupable : 12.6
  Position médiane                  : 15
  Top-1 (rang 1)                    : 3/33
  Top-3 (rang ≤ 3)                  : 8/33
  Top-10 (rang ≤ 10)                : 13/33
  Non trouvé dans le ranking        : 2/33

  Services les plus souvent prédits à tort en top-1 :
    ts-food-service                          : 20 fois
    ts-cancel-service                        : 4 fois
    ts-execute-service                       : 2 fois
    ts-preserve-service                      : 1 fois
    ts-delivery-service                      : 1 fois

Type : exception  (39 fenêtres)
  Position moyenne du vrai coupable : 14.3
  Position médiane                  : 13
  Top-1 (rang 1)                    : 0/39
  Top-3 (rang ≤ 3)                  : 0/39
  Top-10 (rang ≤ 10)                : 8/39
  Non trouvé dans le ranking        : 0/39

  Services les plus souvent prédits à tort en top-1 